What is the motivation for this session? What questions does it answer?

- What do I do when I have a big dataset so I (for example) run out of memory during training?
  - You batch the data.
- How do I batch the data efficiently?
  - Use torch.util.data's DataLoader
    - TensorDataset to create a Dataset class from tensors that can be read by DataLoader and Dataset to create a custom dataset class (advanced topic)
- How should I make sure that the model doesn't just memorize the training data and can't generalize to unseen data?
  - Split into train (and validation) and test data

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

## Section 1: Batching Data to Train Efficiently

| Code | Description |
|---|---|
| `DataLoader(dataset)` | Create a DataLoader for `dataset` and assign it to the variable `dataloader`. |
| `DataLoader(dataset, batch_size = some_num)` | Create a DataLoader for `dataset` where the size of each batch set to `some_num` (default batchsize is 1). |
| `DataLoader(dataset, batch_size = some_num, drop_last = True)` | Create a DataLoader for `dataset` and drop last batch if it's not the same size as the other batches. |
| `DataLoader(dataset, shuffle = True)` | Create a DataLoader for `dataset` and reshuffle the data at each iteration (epoch) during training. |
| `dataloader = DataLoader(dataset)` | Create a DataLoader for `dataset` and assign it to the variable `dataloader`. |

#### **Exercises**

In [56]:
url = 'https://gist.githubusercontent.com/tijptjik/9408623/raw/b237fa5848349a14a14e5d4107dc7897c21951f5/wine.csv'
df = pd.read_csv(url)

# the data in the wine column are the labels
labels = df['Wine'].values

# the data from all other columns are the features
features = df.drop(columns='Wine').values

features = torch.tensor(features, dtype=torch.float32)
labels = torch.tensor(labels, dtype=torch.long)

In [57]:
dataset = TensorDataset(features,labels)

**Example**: Create a dataloader using the TensorDataset variable `dataset` created above. Then run the cell below where the dataloader is looped through and the dimensions of the features in each batch are printed. How big is the sample in each batch?

In [59]:
dataloader = DataLoader(dataset)

In [55]:
n_samples_processed = 0
for idx_batch, (features_batch, labels_batch) in enumerate(dataloader):
    print(f"Batch nr: {idx_batch}, number of samples in batch: {features_batch.shape[0]}, total number of samples looped through: {n_samples_processed}")

    n_samples_processed += features_batch.shape[0]

    if idx_batch > 4:
        break

Batch nr: 0, number of samples in batch: 1, total number of samples looped through: 0
Batch nr: 1, number of samples in batch: 1, total number of samples looped through: 1
Batch nr: 2, number of samples in batch: 1, total number of samples looped through: 2
Batch nr: 3, number of samples in batch: 1, total number of samples looped through: 3
Batch nr: 4, number of samples in batch: 1, total number of samples looped through: 4
Batch nr: 5, number of samples in batch: 1, total number of samples looped through: 5


By default, the batch size in the dataloader is 1. Therefore, the number of samples in each batch in the example above was 1.

**Exercise**: Create a dataloader with batch size 8 and run the cell with the loop.

In [45]:
dataloader = DataLoader(dataset, batch_size=8)

In [46]:
n_samples_processed = 0
for idx_batch, (features_batch, labels_batch) in enumerate(dataloader):
    print(f"Batch nr: {idx_batch}, number of samples in batch: {features_batch.shape[0]}, total number of samples looped through: {n_samples_processed}")

    n_samples_processed += features_batch.shape[0]

    if idx_batch > 5:
        break

Batch nr: 0, number of samples in batch: 8, total number of samples looped through: 0
Batch nr: 1, number of samples in batch: 8, total number of samples looped through: 8
Batch nr: 2, number of samples in batch: 8, total number of samples looped through: 16
Batch nr: 3, number of samples in batch: 8, total number of samples looped through: 24
Batch nr: 4, number of samples in batch: 8, total number of samples looped through: 32
Batch nr: 5, number of samples in batch: 8, total number of samples looped through: 40
Batch nr: 6, number of samples in batch: 8, total number of samples looped through: 48


By default, the batch size in the dataloader is 1. Therefore, the number of samples in each batch in the example above was 1.

**Exercise**: Create a dataloader with batch size `50` and run the cell with the loop.

In [49]:
dataloader = DataLoader(dataset, batch_size=50)

In [50]:
n_samples_processed = 0
for idx_batch, (features_batch, labels_batch) in enumerate(dataloader):
    print(f"Batch nr: {idx_batch}, number of samples in batch: {features_batch.shape[0]}, total number of samples looped through: {n_samples_processed}")

    n_samples_processed += features_batch.shape[0]

    if idx_batch > 5:
        break

Batch nr: 0, number of samples in batch: 50, total number of samples looped through: 0
Batch nr: 1, number of samples in batch: 50, total number of samples looped through: 50
Batch nr: 2, number of samples in batch: 50, total number of samples looped through: 100
Batch nr: 3, number of samples in batch: 28, total number of samples looped through: 150


Notice that the last batch in the previous exercise is of size `28`. That's because the total number of samples in the dataset is 178, and 178-150 = 28. There aren't enough samples left to fill up a batch size of 50 in the last iteration.

**Exercise**: Create the dataloader with a batch size of 50 again, but set `drop_last=True`. What happens to the last batch in the loop?

In [ ]:
dataloader = DataLoader(dataset, batch_size=50, drop_last=True)

In [52]:
n_samples_processed = 0
for idx_batch, (features_batch, labels_batch) in enumerate(dataloader):
    print(f"Batch nr: {idx_batch}, number of samples in batch: {features_batch.shape[0]}, total number of samples looped through: {n_samples_processed}")

    n_samples_processed += features_batch.shape[0]

    if idx_batch > 5:
        break

Batch nr: 0, number of samples in batch: 50, total number of samples looped through: 0
Batch nr: 1, number of samples in batch: 50, total number of samples looped through: 50
Batch nr: 2, number of samples in batch: 50, total number of samples looped through: 100


**Exercise**: Run the cell below where a dataloader is created with the default batch size and without shuffling. Do you get the same output when you run it again?

In [68]:
dataloader = DataLoader(dataset)

feature, label = next(iter(dataloader))
feature, label

(tensor([[1.4230e+01, 1.7100e+00, 2.4300e+00, 1.5600e+01, 1.2700e+02, 2.8000e+00,
          3.0600e+00, 2.8000e-01, 2.2900e+00, 5.6400e+00, 1.0400e+00, 3.9200e+00,
          1.0650e+03]]),
 tensor([1]))

You should have gotten the same feature values and label in the previous exercise no matter how many times you reran the code cell.

**Exercise**: Create a dataloader for the dataset again, but this time, set the parameter `shuffle = True`. Then run the cell below that gets a set of features and their label from the dataset. Do you get the same values when you run it repeatedly now?

In [70]:
dataloader = DataLoader(dataset, shuffle=True)

In [73]:
feature, label = next(iter(dataloader))
feature, label

(tensor([[ 12.4200,   2.5500,   2.2700,  22.0000,  90.0000,   1.6800,   1.8400,
            0.6600,   1.4200,   2.7000,   0.8600,   3.3000, 315.0000]]),
 tensor([2]))

## Section 2: Train with a Dataloader

In [ ]:
class Model(nn.Module):
    def __init__(self,):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(13, 16),
            nn.ReLU(),
            nn.Linear(16,13)
        )

    def forward(self, x):
        output = self.layers(x)
        return output
    
Model()

Model(
  (layers): Sequential(
    (0): Linear(in_features=13, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=13, bias=True)
  )
)

## Section 3: Train Test Splitting

## Section 4: Creating Custom Dataset Classes